In [1]:
import pandas as pd
import numpy as np

In [2]:
data = []
with open('f1.fasta', "r") as f:
    for line in f:
        line = line.strip()
        if line.startswith(">"):
            continue
        data.append(line)

f1_sequence = "".join(data)

In [3]:
def get_kmers(sequence, k):
    kmers = []
    for i in range(len(sequence) - k + 1):
        kmer = sequence[i : i + k]
        kmers.append(kmer)
    return kmers
kmers=get_kmers(f1_sequence,32)
f1=pd.DataFrame(kmers)
f1

,0
0,GACAGGTACAAGAAGGAGTATGGATCAATCTG
1,ACAGGTACAAGAAGGAGTATGGATCAATCTGG
2,CAGGTACAAGAAGGAGTATGGATCAATCTGGT
3,AGGTACAAGAAGGAGTATGGATCAATCTGGTC
4,GGTACAAGAAGGAGTATGGATCAATCTGGTCG
...,...
49964,TTTCATTCACTTGATGTCCTTAACGATAGCCA
49965,TTCATTCACTTGATGTCCTTAACGATAGCCAA
49966,TCATTCACTTGATGTCCTTAACGATAGCCAAC
49967,CATTCACTTGATGTCCTTAACGATAGCCAACC


In [4]:
data = []
with open('p1.fasta', "r") as f:
    for line in f:
        line = line.strip()
        if line.startswith(">"):
            continue
        data.append(line)

p1_sequence = "".join(data)
kmers=get_kmers(p1_sequence,8)
p1=pd.DataFrame(kmers)
p1=p1[0]
p1=p1.unique()
p1

array(['GACAGGTA', 'ACAGGTAC', 'CAGGTACA', ..., 'ATAGCCAA', 'TAGCCAAC',
       'GCCAACCG'], dtype=object)

In [5]:
data = []
with open('p2.fasta', "r") as f:
    for line in f:
        line = line.strip()
        if line.startswith(">"):
            continue
        data.append(line)

p2_sequence = "".join(data)
kmers=get_kmers(p2_sequence,8)
p2=pd.DataFrame(kmers)
p2=p2[0]
p2=p2.unique()
p2

array(['GACAGGTA', 'ACAGGTAC', 'CAGGTACA', ..., 'GGCCTTAA', 'GCCTTAAG',
       'GGCACCCG'], dtype=object)

In [6]:
f1

,0
0,GACAGGTACAAGAAGGAGTATGGATCAATCTG
1,ACAGGTACAAGAAGGAGTATGGATCAATCTGG
2,CAGGTACAAGAAGGAGTATGGATCAATCTGGT
3,AGGTACAAGAAGGAGTATGGATCAATCTGGTC
4,GGTACAAGAAGGAGTATGGATCAATCTGGTCG
...,...
49964,TTTCATTCACTTGATGTCCTTAACGATAGCCA
49965,TTCATTCACTTGATGTCCTTAACGATAGCCAA
49966,TCATTCACTTGATGTCCTTAACGATAGCCAAC
49967,CATTCACTTGATGTCCTTAACGATAGCCAACC


In [7]:
# Step 1 - read and score
def total_matches(x):
    f1_kmers = get_kmers(x, 8)
    total = len(f1_kmers)
    p1_hits = len(set(f1_kmers).intersection(p1))
    return p1_hits / total

f1['matches'] = f1[0].apply(total_matches)

# Step 2 - observation label
f1['observation'] = f1['matches'].apply(
    lambda x: "p1" if x > 0.7 else "p2" if x < 0.3 else "amb"
)

# Step 3 - pseudo state label
f1['pseudo_state'] = f1['matches'].apply(
    lambda x: "UP1" if x > 0.9 else "UP2" if x < 0.1 else "S"
)

# Sanity check
print(f1.columns.tolist())
print(f1['observation'].value_counts())
print(f1['pseudo_state'].value_counts())

[0, 'matches', 'observation', 'pseudo_state']
observation
p1     33392
amb    16397
p2       180
Name: count, dtype: int64
pseudo_state
S      25135
UP1    24834
Name: count, dtype: int64


In [8]:
f1

,0,matches,observation,pseudo_state
0,GACAGGTACAAGAAGGAGTATGGATCAATCTG,0.68,amb,S
1,ACAGGTACAAGAAGGAGTATGGATCAATCTGG,0.64,amb,S
2,CAGGTACAAGAAGGAGTATGGATCAATCTGGT,0.60,amb,S
3,AGGTACAAGAAGGAGTATGGATCAATCTGGTC,0.56,amb,S
4,GGTACAAGAAGGAGTATGGATCAATCTGGTCG,0.52,amb,S
...,...,...,...,...
49964,TTTCATTCACTTGATGTCCTTAACGATAGCCA,1.00,p1,UP1
49965,TTCATTCACTTGATGTCCTTAACGATAGCCAA,1.00,p1,UP1
49966,TCATTCACTTGATGTCCTTAACGATAGCCAAC,1.00,p1,UP1
49967,CATTCACTTGATGTCCTTAACGATAGCCAACC,1.00,p1,UP1


In [9]:
def calc_emissions(df):
    emissions = np.zeros((3, 3))  # [states x observations]
    
    for i, state in enumerate(["UP1", "UP2", "S"]):
        subset = df[df['pseudo_state'] == state]
        total = len(subset)
        if total == 0:
            continue
        emissions[i, 0] = (subset['observation'] == 'p1').sum()  / total
        emissions[i, 1] = (subset['observation'] == 'p2').sum()  / total
        emissions[i, 2] = (subset['observation'] == 'amb').sum() / total
    
    return emissions

In [10]:
# Step 1 - MUST run this first
def pseudo_label(x):
    if x > 0.9:
        return "UP1"
    elif x < 0.1:
        return "UP2"
    else:
        return "S"

f1['pseudo_state'] = f1['matches'].apply(pseudo_label)

# Quick check before running transitions
print(f1['pseudo_state'].value_counts())
# Should show UP1, UP2, S  --- NOT p1, p2, amb

pseudo_state
S      25135
UP1    24834
Name: count, dtype: int64


In [11]:
def calc_transitions(df):
    transitions = np.zeros((3, 3))  # [from x to]
    state_index = {"UP1": 0, "UP2": 1, "S": 2}
    
    states = df['pseudo_state'].values
    for i in range(len(states) - 1):
        from_state = state_index[states[i]]
        to_state   = state_index[states[i + 1]]
        transitions[from_state, to_state] += 1
    
    # normalize rows to sum to 1
    row_sums = transitions.sum(axis=1, keepdims=True)
    transitions = transitions / row_sums
    
    return transitions

emissions   = calc_emissions(f1)
transitions = calc_transitions(f1)

print("Emissions [UP1, UP2, S] x [p1, p2, amb]:")
print(emissions)
print("\nTransitions [UP1, UP2, S] x [UP1, UP2, S]:")
print(transitions)

Emissions [UP1, UP2, S] x [p1, p2, amb]:
[[1.         0.         0.        ]
 [0.         0.         0.        ]
 [0.3404814  0.00716133 0.65235727]]

Transitions [UP1, UP2, S] x [UP1, UP2, S]:
[[0.9953288  0.         0.0046712 ]
 [       nan        nan        nan]
 [0.00465486 0.         0.99534514]]


/var/folders/n_/1hzw47553v1bvg9z8kct6rd40000gn/T/ipykernel_87985/15328268.py:13: RuntimeWarning: invalid value encountered in divide
  transitions = transitions / row_sums


In [12]:
# Transition matrix - rows=from, cols=to
# order: [UP1, UP2, S]
transitions = np.array([
    #  UP1   UP2    S
    [0.90, 0.05, 0.05],   # from UP1 → stays in UP1 most of the time
    [0.05, 0.90, 0.05],   # from UP2 → stays in UP2 most of the time
    [0.45, 0.45, 0.10],   # from S   → quickly resolves to one parent
])

# Emission matrix - rows=state, cols=observation
# order: [p1, p2, amb]
emissions = np.array([
    #   p1    p2   amb
    [0.90, 0.05, 0.05],   # UP1 → almost always matches P1
    [0.05, 0.90, 0.05],   # UP2 → almost never matches P1
    [0.25, 0.25, 0.50],   # S   → genuinely ambiguous
])

obs_index   = {"p1": 0, "p2": 1, "amb": 2}
state_index = {"UP1": 0, "UP2": 1, "S": 2}
states      = ["UP1", "UP2", "S"]

# Sanity check
assert np.allclose(transitions.sum(axis=1), 1.0)
assert np.allclose(emissions.sum(axis=1), 1.0)

In [13]:
def viterbi(observations, transitions, emissions, states, obs_index):
    n_states = len(states)
    n_obs    = len(observations)
    
    # Initialize
    viterbi_matrix = np.zeros((n_states, n_obs))
    backpointer    = np.zeros((n_states, n_obs), dtype=int)
    
    # Step 1 - initialize with equal starting probability
    initial = np.array([1/n_states] * n_states)
    obs0    = obs_index[observations[0]]
    for s in range(n_states):
        viterbi_matrix[s, 0] = initial[s] * emissions[s, obs0]
    
    # Step 2 - fill in the rest
    for t in range(1, n_obs):
        obs_t = obs_index[observations[t]]
        for s in range(n_states):
            # probability of arriving at state s from each previous state
            probs = [
                viterbi_matrix[prev_s, t-1] * transitions[prev_s, s] * emissions[s, obs_t]
                for prev_s in range(n_states)
            ]
            viterbi_matrix[s, t] = max(probs)
            backpointer[s, t]    = np.argmax(probs)
    
    # Step 3 - trace back the best path
    path = np.zeros(n_obs, dtype=int)
    path[-1] = np.argmax(viterbi_matrix[:, -1])
    for t in range(n_obs - 2, -1, -1):
        path[t] = backpointer[path[t+1], t+1]
    
    return [states[s] for s in path]


# Run it
obs_index = {"p1": 0, "p2": 1, "amb": 2}
obs_sequence = f1['observation'].tolist()

decoded_path = viterbi(
    observations = obs_sequence,
    transitions  = transitions,
    emissions    = emissions,
    states       = ["UP1", "UP2", "S"],
    obs_index    = obs_index
)

f1['predicted_state'] = decoded_path
print(f1['predicted_state'].value_counts())

predicted_state
UP1    49722
S        235
UP2       12
Name: count, dtype: int64
